# Query Due Cards from the JP Deck

This notebook connects to Anki through AnkiConnect and retrieves up to the first 100 due cards from the `JP` deck.

Why this version is safer and more reliable:
- It uses AnkiConnect instead of reading `collection.anki2` directly.
- It asks Anki for `is:due`, which matches Anki's own scheduler semantics better than raw SQL.
- It is structured as a valid Jupyter notebook instead of a single pasted code cell.

In [48]:
import json
from urllib import request

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [49]:
ANKI_CONNECT_URL = "http://127.0.0.1:8765"
ANKI_CONNECT_VERSION = 6

DECK_NAME = "JP"
LIMIT = 100

In [50]:
def invoke(action, **params):
    payload = {
        "action": action,
        "version": ANKI_CONNECT_VERSION,
        "params": params,
    }
    req = request.Request(
        ANKI_CONNECT_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with request.urlopen(req, timeout=30) as response:
        data = json.load(response)

    if data.get("error"):
        raise RuntimeError(f"AnkiConnect error in {action}: {data['error']}")

    return data["result"]


print("Connected AnkiConnect version:", invoke("version"))

Connected AnkiConnect version: 6


In [51]:
def field_value(card, field_name):
    return card.get("fields", {}).get(field_name, {}).get("value", "")


query = f'deck:{DECK_NAME} prop:ivl<=7 is:review -tag:marked -is:suspended'
card_ids = invoke("findCards", query=query)
cards = invoke("cardsInfo", cards=card_ids) if card_ids else []

# Sort defensively in case AnkiConnect returns cards in search order instead of due order.
cards = sorted(cards, key=lambda card: (card.get("due") is None, card.get("due", 0), card["cardId"]))[:LIMIT]

rows = []
for card in cards:
    rows.append(
        {
            "card_id": card["cardId"],
            "note_id": card["note"],
            "deck": card.get("deckName", ""),
            "due": card.get("due"),
            "queue": card.get("queue"),
            "type": card.get("type"),
            "lemma": field_value(card, "Lemma"),
            "cloze": field_value(card, "Cloze"),
            "definition": field_value(card, "Word Definition"),
        }
    )

due_cards = pd.DataFrame(rows)
print(f"Found {len(due_cards)} due card(s) for query: {query}")

Found 100 due card(s) for query: deck:JP prop:ivl<=7 is:review -tag:marked -is:suspended


In [52]:
if due_cards.empty:
    print(f"No due cards found for deck '{DECK_NAME}'.")
else:
    display(due_cards)

,card_id,note_id,deck,due,queue,type,lemma,cloze,definition
0,1739865122145,1739865122145,JP,1423,2,2,退屈,"‎{{c1::退屈::Langeweile, Öde, Eintönigkeit}}だった？","langweilig, eintönig"
1,1770076175755,1770076175754,JP,1423,2,2,苦痛,するなんて{{c1::苦痛::fürchterlich}}じゃないかって,"pain, suffering"
2,1771350265960,1771350265959,JP,1423,2,2,持ち主,"残りは2、3日中に渡す {{c1::持ち主::Besitzer, Inhaber}}はこれだ","Eigentümer, Inhaber, Besitzer"
3,1772280083744,1772280083744,JP,1423,2,2,転ぶ,ことが大事かな。{{c1::転\n\nんでもいいんすよ。&nbsp;::Man darf a...,"fallen, stolpern"
4,1772362242539,1772362242539,JP,1423,2,2,停止,"春と夏の方も安全確認のために1時間ほど{{c1::停止::Stillstand, Unter...","Stillstand, Unterbrechung"
...,...,...,...,...,...,...,...,...,...
95,1774960701698,1774960701698,JP,1427,2,2,見逃す,ねすごいでしょだから今日は{{c1::<strong>見逃して</strong>::über...,"übersehen, durchgehen lassen"
96,1772279893071,1772279893071,JP,1428,2,2,向き合う,"{{c1::向き合う::konfrontieren, gegenüberstehen}}いく...","konfrontieren, gegenüberstehen"
97,1772429873467,1772429873467,JP,1428,2,2,泊まる,"親父さんぼくの{{c1::泊まる::übernachten, bleiben}}とこ<br>...","übernachten, bleiben"
98,1772463655895,1772463655895,JP,1428,2,2,両方,"ここはひとつ<br>{{c1::両方::beide, beide Seiten}}の顔をたててだな","beide, beide Seiten"


In [53]:
import subprocess

if due_cards.empty:
    print("No lemmas to display or copy.")
else:
    lemma_series = due_cards["lemma"].fillna("").astype(str).str.strip()
    lemma_series = lemma_series[lemma_series != ""]
    lemma_text = "\n".join(lemma_series.tolist())

    display(pd.DataFrame({"lemma": lemma_series.tolist()}))
    print(lemma_text)

    try:
        subprocess.run(["pbcopy"], input=lemma_text, text=True, check=True)
        print("\nCopied lemma list to clipboard.")
    except FileNotFoundError:
        print("\npbcopy is not available in this environment, so the list was not copied automatically.")


,lemma
0,退屈
1,苦痛
2,持ち主
3,転ぶ
4,停止
...,...
95,見逃す
96,向き合う
97,泊まる
98,両方


退屈
苦痛
持ち主
転ぶ
停止
ほっと
応じる
魅力
乾く
くっつく
訪ねる
収まる
代わる
手前
乗り越える
生命
変更
飽きる
乱れる
始末
発揮
現状
認識
所属
維持
範囲
自宅
設定
拭く
根
<strong>恋文</strong>
共作
いっそ
行為
引きずる
被る
爪
治る
資料
強力
指す
よほど
仮に
指摘
移す
整う
積む
頂戴
あっという間に
強烈
もたらす
とっくに
眺める
用事
賭け事
なるべく
約
計算
いかん
やる気
都合
狂う
掲げる
直前
正式
してやる
段階
どうにも
崩れる
溶ける
代表
縁
屋根
以降
湧く
縛る
実感
両
日常
きく
伏せる
逆らう
意図
その間
乱暴
自分自身
負う
どうやら
略
一旦
飾る
詰まる
さっぱり
複数
嫉妬
見逃す
向き合う
泊まる
両方
いかに

Copied lemma list to clipboard.
